In [3]:
!pip install pandas numpy nltk spacy scikit-learn pdfplumber

In [4]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 98.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
import os
import re
import pdfplumber
import pandas as pd
import nltk
import spacy

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [7]:
nlp = spacy.load("en_core_web_sm")

In [31]:
from google.colab import files

uploaded = files.upload()

Saving resume2.pdf to resume2 (2).pdf
Saving resume3.pdf to resume3 (1).pdf
Saving resume1.pdf to resume1 (1).pdf


In [10]:
SKILLS = [
    "python",
    "java",
    "c++",
    "machine learning",
    "deep learning",
    "nlp",
    "sql",
    "mysql",
    "mongodb",
    "tensorflow",
    "keras",
    "pytorch",
    "data science",
    "data analysis",
    "pandas",
    "numpy",
    "scikit-learn",
    "excel",
    "power bi",
    "tableau",
    "aws",
    "docker",
    "git",
    "flask",
    "django"
]

In [11]:
stop_words = set(stopwords.words('english'))

In [32]:
def extract_text_from_pdf(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            extracted = page.extract_text()

            if extracted:
                text += extracted + " "

    return text

In [33]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    return " ".join(words)

In [34]:
def extract_skills(text):

    found_skills = []

    text = text.lower()

    for skill in SKILLS:

        if skill.lower() in text:
            found_skills.append(skill)

    return list(set(found_skills))

In [35]:
def calculate_similarity(resume_text, job_description):

    documents = [resume_text, job_description]

    tfidf = TfidfVectorizer()

    tfidf_matrix = tfidf.fit_transform(documents)

    similarity = cosine_similarity(
        tfidf_matrix[0:1],
        tfidf_matrix[1:2]
    )

    return round(similarity[0][0] * 100, 2)

In [36]:
def find_missing_skills(candidate_skills, jd_skills):

    missing = []

    for skill in jd_skills:

        if skill not in candidate_skills:
            missing.append(skill)

    return missing

In [37]:
job_description = """
We are looking for a Data Scientist with experience in
Python, Machine Learning, NLP, SQL, TensorFlow,
Scikit-learn, Pandas, NumPy, AWS, and Docker.
"""

In [38]:
cleaned_jd = clean_text(job_description)

jd_skills = extract_skills(cleaned_jd)

print("Required Skills:")
print(jd_skills)

Required Skills:
['machine learning', 'pandas', 'nlp', 'sql', 'python', 'aws', 'docker', 'tensorflow', 'numpy']


In [39]:
results = []

for file_name in uploaded.keys():

    print(f"\nProcessing Resume: {file_name}")

    raw_text = extract_text_from_pdf(file_name)

    cleaned_resume = clean_text(raw_text)

    candidate_skills = extract_skills(cleaned_resume)

    score = calculate_similarity(
        cleaned_resume,
        cleaned_jd
    )

    missing_skills = find_missing_skills(
        candidate_skills,
        jd_skills
    )

    results.append({
        "Candidate": file_name,
        "Score": score,
        "Skills": ", ".join(candidate_skills),
        "Missing Skills": ", ".join(missing_skills)
    })


Processing Resume: resume2 (2).pdf

Processing Resume: resume3 (1).pdf

Processing Resume: resume1 (1).pdf


In [40]:
results_df = pd.DataFrame(results)

In [41]:
results_df = results_df.sort_values(
    by="Score",
    ascending=False
)

In [42]:
print("\nRANKED CANDIDATES")
print(results_df)


RANKED CANDIDATES
         Candidate  Score                                             Skills  \
2  resume1 (1).pdf  35.63  machine learning, pandas, sql, python, aws, te...   
1  resume3 (1).pdf  27.30  machine learning, nlp, deep learning, pytorch,...   
0  resume2 (2).pdf  12.24  data analysis, sql, java, excel, power bi, docker   

                                      Missing Skills  
2                                        nlp, docker  
1                pandas, sql, aws, tensorflow, numpy  
0  machine learning, pandas, nlp, python, aws, te...  


In [43]:
results_df.to_csv("ranked_candidates.csv", index=False)

print("Results saved successfully")

Results saved successfully


In [44]:
from google.colab import files

files.download("ranked_candidates.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>